Analyzation of the sample data (Not the entire dataset)

In [497]:
import pandas as pd
import numpy as np

In [498]:
df_1d_edge_index = pd.read_csv("early_data/1d_edge_index.csv")
df_1d_edges_dynamic_all = pd.read_csv("early_data/1d_edges_dynamic_all.csv")
df_1d_edges_static = pd.read_csv("early_data/1d_edges_static.csv")
df_1d_nodes_dynamic_all = pd.read_csv("early_data/1d_nodes_dynamic_all.csv")
df_1d_nodes_static = pd.read_csv("early_data/1d_nodes_static.csv")
df_1d2d_connections = pd.read_csv("early_data/1d2d_connections.csv")

df_2d_edge_index = pd.read_csv("early_data/2d_edge_index.csv")
df_2d_edges_dynamic_all = pd.read_csv("early_data/2d_edges_dynamic_all.csv")
df_2d_edges_static = pd.read_csv("early_data/2d_edges_static.csv")
df_2d_nodes_dynamic_all = pd.read_csv("early_data/2d_nodes_dynamic_all.csv")
df_2d_nodes_static = pd.read_csv("early_data/2d_nodes_static.csv")
timesteps = pd.read_csv("early_data/timesteps.csv")

Quick analyzation of each csv, script generated with AI.

In [499]:
dfs = {
    "df_1d_edge_index": df_1d_edge_index,
    "df_1d_edges_dynamic_all": df_1d_edges_dynamic_all,
    "df_1d_edges_static": df_1d_edges_static,
    "df_1d_nodes_dynamic_all": df_1d_nodes_dynamic_all,
    "df_1d_nodes_static": df_1d_nodes_static,
    "df_1d2d_connections": df_1d2d_connections,
    "df_2d_edge_index": df_2d_edge_index,
    "df_2d_edges_dynamic_all": df_2d_edges_dynamic_all,
    "df_2d_edges_static": df_2d_edges_static,
    "df_2d_nodes_dynamic_all": df_2d_nodes_dynamic_all,
    "df_2d_nodes_static": df_2d_nodes_static,
    "timesteps": timesteps,
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

LIKELY_ID_COLS = [
    "edge_id", "edge_idx", "edge_index", "u", "v", "src", "dst", "from", "to",
    "node_id", "node_idx", "cell_id", "face_id",
    "link_id", "reach_id", "seg_id",
    "t", "time", "timestamp", "timestep", "step",
]

for name, df in dfs.items():
    print("\n" + "=" * 110)
    print(f"{name}")
    print("-" * 110)

    # shape
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]:,} cols")
    # columns + dtypes (compact)
    dtypes_counts = df.dtypes.value_counts()
    print(f"dtypes: " + ", ".join([f"{k}={v}" for k, v in dtypes_counts.items()]))

    # missingness
    na_total = int(df.isna().sum().sum())
    na_cols = int((df.isna().sum() > 0).sum())
    print(f"missing values: {na_total:,} total across {na_cols}/{df.shape[1]} columns")

    # duplicate rows
    dup = int(df.duplicated().sum())
    print(f"duplicate rows: {dup:,}")

    # show first few rows (small peek)
    print("\nhead(3):")
    print(df.head(3))

    # likely key columns: unique counts + basic min/max for numeric
    present = [c for c in LIKELY_ID_COLS if c in df.columns]
    if present:
        print("\nlikely key columns present:")
        for c in present:
            s = df[c]
            nun = s.nunique(dropna=True)
            nmiss = int(s.isna().sum())
            print(f"  - {c}: nunique={nun:,} missing={nmiss:,}", end="")

            if pd.api.types.is_numeric_dtype(s):
                s2 = s.dropna()
                if len(s2):
                    print(f" | min={s2.min()} max={s2.max()}")
                else:
                    print()
            else:
                # show a few example values
                vals = s.dropna().astype(str).unique()[:5]
                print(f" | examples={list(vals)}")
    else:
        print("\n(no obvious id/time columns found from the common list)")

    # numeric summary (only if there are numeric cols)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\nnumeric columns summary (count/mean/std/min/max) [first 25 numeric cols]:")
        show = num_cols[:25]
        desc = df[show].describe().loc[["count", "mean", "std", "min", "max"]]
        print(desc)

    # string/object quick check
    obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if obj_cols:
        show = obj_cols[:10]
        print("\nobject columns unique counts [first 10 object cols]:")
        for c in show:
            nun = df[c].nunique(dropna=True)
            print(f"  - {c}: nunique={nun:,}")

print("\n" + "=" * 110)
print("Done.")



df_1d_edge_index
--------------------------------------------------------------------------------------------------------------
shape: 16 rows x 3 cols
dtypes: int64=3
missing values: 0 total across 0/3 columns
duplicate rows: 0

head(3):
   edge_idx  from_node  to_node
0         0          0       16
1         1         13       12
2         2          3       13

likely key columns present:
  - edge_idx: nunique=16 missing=0 | min=0 max=15

numeric columns summary (count/mean/std/min/max) [first 25 numeric cols]:
        edge_idx  from_node    to_node
count  16.000000  16.000000  16.000000
mean    7.500000   7.500000   6.500000
std     4.760952   4.760952   5.773503
min     0.000000   0.000000   0.000000
max    15.000000  15.000000  16.000000

df_1d_edges_dynamic_all
--------------------------------------------------------------------------------------------------------------
shape: 1,504 rows x 4 cols
dtypes: int64=2, float64=2
missing values: 0 total across 0/4 columns
duplicate r

Huge outlier in 2D nodes 'area' feature. -1.598721e-14 shouldn't be there. It also has Nans.

In [500]:
df_2d_nodes_static['area'] = df_2d_nodes_static['area'].clip(0)

In [501]:
mask_nan = df_2d_nodes_static["min_elevation"].isna()
df_2d_nodes_static.loc[mask_nan, "min_elevation"] = df_2d_nodes_static.loc[mask_nan, "elevation"]

Check if there are any rows where min_elevation > elevation

In [502]:
(df_2d_nodes_static['elevation'][0:] - df_2d_nodes_static['min_elevation'][0:]).describe()

count    3716.000000
mean        1.305382
std         1.240858
min        -0.006320
25%         0.432318
50%         0.860775
75%         1.779740
max         8.569550
dtype: float64

In [503]:
mask = df_2d_nodes_static['min_elevation'] > df_2d_nodes_static['elevation']
df_2d_nodes_static.loc[mask, 'min_elevation'] = df_2d_nodes_static.loc[mask, 'elevation']

Plan is to inject nearby node info into the hidden state of whatever model I use. Find all neighbors of each 1D node.

In [504]:
adj_1d = []
for i in range(df_1d_nodes_static["node_idx"].max() + 1):
    adj_1d.append([])
adj_1d

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]

In [505]:
for fr, to in zip(df_1d_edge_index['from_node'], df_1d_edge_index['to_node']):
    adj_1d[fr].append(to)
    adj_1d[to].append(fr)

adj_1d

[[16, 6, 8],
 [8, 9, 10, 2],
 [1, 11],
 [13, 4],
 [3, 14],
 [14, 15],
 [0],
 [13],
 [0, 1],
 [1],
 [1],
 [2, 12],
 [13, 11],
 [12, 3, 7],
 [4, 5],
 [5],
 [0]]

Same for 2D

In [506]:
adj_2d = []
for i in range(df_2d_nodes_static["node_idx"].max() + 1):
    adj_2d.append([])

for fr, to in zip(df_2d_edge_index['from_node'], df_2d_edge_index['to_node']):
    adj_2d[fr].append(to)
    adj_2d[to].append(fr)

print(adj_2d[0:5])
len(adj_2d)

[[14, 1, 13], [0, 15, 2], [1, 16, 3], [2, 17, 4], [3, 18, 5]]


3716

Now 2D to 1D and vice versa

In [507]:
cpl_2d_to_1d = {}
cpl_1d_to_2d = {}

for one, two in zip(df_1d2d_connections['node_1d'], df_1d2d_connections['node_2d']):
    cpl_1d_to_2d[one] = [two]
    cpl_2d_to_1d[two] = [one]
cpl_1d_to_2d

{0: [503],
 13: [3669],
 12: [3588],
 3: [3595],
 4: [3600],
 14: [2378],
 5: [2262],
 15: [2152],
 7: [3241],
 6: [551],
 8: [600],
 1: [2960],
 9: [2939],
 10: [2958],
 2: [2952],
 11: [1503]}

2D features

In [508]:
EPS = 1e-6

n2 = int(df_2d_nodes_static["node_idx"].max()) + 1

elev2 = np.full(n2, np.nan, np.float32)
elev2[df_2d_nodes_static["node_idx"].to_numpy(np.int64)] = df_2d_nodes_static["elevation"].to_numpy(np.float32)

rough2 = np.full(n2, np.nan, np.float32)
rough2[df_2d_nodes_static["node_idx"].to_numpy(np.int64)] = df_2d_nodes_static["roughness"].to_numpy(np.float32)

E2u = df_2d_edge_index.merge(df_2d_edges_static, on="edge_idx", how="left", validate="one_to_one")

u2 = E2u["from_node"].to_numpy(np.int64)
v2 = E2u["to_node"].to_numpy(np.int64)

face2 = E2u["face_length"].to_numpy(np.float32)
len2  = E2u["length"].to_numpy(np.float32)

eu2 = elev2[u2]
ev2 = elev2[v2]
m2 = np.isfinite(eu2) & np.isfinite(ev2) & np.isfinite(face2) & np.isfinite(len2)

u2 = u2[m2]; v2 = v2[m2]
face2 = face2[m2]; len2 = len2[m2]
eu2 = eu2[m2]; ev2 = ev2[m2]

swap2 = ev2 > eu2 # if v is higher, make v -> u
src2 = np.where(swap2, v2, u2)
dst2 = np.where(swap2, u2, v2)

dz2 = elev2[src2] - elev2[dst2]
slopez2 = dz2 / (len2 + EPS)

# roughness along edge = mean endpoint roughness
r2 = 0.5 * (rough2[src2] + rough2[dst2])
r2 = np.where(np.isfinite(r2), r2, 0.06).astype(np.float32)  # fallback to typical roughness if needed

# conductance-like weight for 2D
# w = (face/len) * (slopez) / roughness
w2_cond = (face2 / (len2 + EPS)) * (slopez2 + EPS) / (r2 + EPS)

# optional alternate weight using provided edge slope magnitude too
slope2_abs = np.abs(E2u["slope"].to_numpy(np.float32)[m2])
w2_cond_slope_static = w2_cond * (slope2_abs + EPS)

1D features

In [509]:
n1 = int(df_1d_nodes_static["node_idx"].max()) + 1

inv1 = np.full(n1, np.nan, np.float32)
inv1[df_1d_nodes_static["node_idx"].to_numpy(np.int64)] = df_1d_nodes_static["invert_elevation"].to_numpy(np.float32)

E1u = df_1d_edge_index.merge(df_1d_edges_static, on="edge_idx", how="left", validate="one_to_one")

u1 = E1u["from_node"].to_numpy(np.int64)
v1 = E1u["to_node"].to_numpy(np.int64)

diam1 = E1u["diameter"].to_numpy(np.float32)
len1 = E1u["length"].to_numpy(np.float32)
rough1 = E1u["roughness"].to_numpy(np.float32)

eu1 = inv1[u1]
ev1 = inv1[v1]
m1 = np.isfinite(eu1) & np.isfinite(ev1) & np.isfinite(diam1) & np.isfinite(len1) & np.isfinite(rough1)

u1 = u1[m1]; v1 = v1[m1]
diam1 = diam1[m1]; len1 = len1[m1]; rough1 = rough1[m1]
eu1 = eu1[m1]; ev1 = ev1[m1]

swap1 = ev1 > eu1
src1 = np.where(swap1, v1, u1)
dst1 = np.where(swap1, u1, v1)

dz1 = inv1[src1] - inv1[dst1]
slopez1 = dz1 / (len1 + EPS)

# conductance-like weight for 1D
# w = (diam^2 / (len * rough)) * slopez
w1_cond = ((diam1 * diam1) / ((len1 + EPS) * (rough1 + EPS))) * (slopez1 + EPS)

# optional alternate weight using provided edge slope magnitude too
slope1_abs = np.abs(E1u["slope"].to_numpy(np.float32)[m1])
w1_cond_slope_static = w1_cond * (slope1_abs + EPS)

In [510]:
def edge_in_out_aggs(wl, src, dst, w, n_nodes):
    """
    For directed edges src -> dst:

    diff = wl[src] - wl[dst]

    Incoming to node i (dst == i):
      in_wmean_wl[i] = sum(w * wl[src]) / sum(w)
      in_flux[i]     = sum(w * diff)

    Outgoing from node i (src == i):
      out_wmean_wl[i] = sum(w * wl[dst]) / sum(w)
      out_flux[i]     = sum(w * diff)

    Net flux proxy at node i:
      net_flux[i] = in_flux[i] - out_flux[i]
    """
    wl_src = wl[src]
    wl_dst = wl[dst]

    m = np.isfinite(wl_src) & np.isfinite(wl_dst) & np.isfinite(w)
    src = src[m]; dst = dst[m]; w = w[m]
    wl_src = wl_src[m]; wl_dst = wl_dst[m]

    diff = wl_src - wl_dst

    # incoming indexed by dst
    in_wsum = np.bincount(dst, weights=w, minlength=n_nodes).astype(np.float32)
    in_wwl  = np.bincount(dst, weights=w * wl_src, minlength=n_nodes).astype(np.float32)
    in_flux = np.bincount(dst, weights=w * diff, minlength=n_nodes).astype(np.float32)
    in_wmean_wl = in_wwl / (in_wsum + EPS)

    # outgoing indexed by src
    out_wsum = np.bincount(src, weights=w, minlength=n_nodes).astype(np.float32)
    out_wwl  = np.bincount(src, weights=w * wl_dst, minlength=n_nodes).astype(np.float32)
    out_flux = np.bincount(src, weights=w * diff, minlength=n_nodes).astype(np.float32)
    out_wmean_wl = out_wwl / (out_wsum + EPS)

    net_flux = in_flux - out_flux
    return in_wmean_wl, out_wmean_wl, in_flux, out_flux, net_flux, in_wsum, out_wsum

In [511]:
def snapshot_at_t(t):
    # dynamic slices
    d1 = df_1d_nodes_dynamic_all[df_1d_nodes_dynamic_all["timestep"] == t].copy()
    d2 = df_2d_nodes_dynamic_all[df_2d_nodes_dynamic_all["timestep"] == t].copy()

    # merge static
    d1 = d1.merge(df_1d_nodes_static, on="node_idx", how="left")
    d2 = d2.merge(df_2d_nodes_static, on="node_idx", how="left")

    # dense water level arrays indexed by node_idx
    wl1 = np.full(n1, np.nan, np.float32)
    wl2 = np.full(n2, np.nan, np.float32)

    wl1[d1["node_idx"].to_numpy(np.int64)] = d1["water_level"].to_numpy(np.float32)
    wl2[d2["node_idx"].to_numpy(np.int64)] = d2["water_level"].to_numpy(np.float32)

    idx1 = d1["node_idx"].to_numpy(np.int64)
    idx2 = d2["node_idx"].to_numpy(np.int64)

    # ---------- 1D edge features (conductance downhill) ----------
    in_wl, out_wl, in_fx, out_fx, net_fx, in_ws, out_ws = edge_in_out_aggs(wl1, src1, dst1, w1_cond, n1)
    d1["e_in_wmean_wl_1d_cond"]  = in_wl[idx1]
    d1["e_out_wmean_wl_1d_cond"] = out_wl[idx1]
    d1["e_in_flux_1d_cond"]      = in_fx[idx1]
    d1["e_out_flux_1d_cond"]     = out_fx[idx1]
    d1["e_net_flux_1d_cond"]     = net_fx[idx1]
    d1["e_in_wsum_1d_cond"]      = in_ws[idx1]
    d1["e_out_wsum_1d_cond"]     = out_ws[idx1]

    # optional: include variant that mixes in static edge slope magnitude
    in_wl, out_wl, in_fx, out_fx, net_fx, in_ws, out_ws = edge_in_out_aggs(wl1, src1, dst1, w1_cond_slope_static, n1)
    d1["e_in_flux_1d_cond_slope"]  = in_fx[idx1]
    d1["e_out_flux_1d_cond_slope"] = out_fx[idx1]
    d1["e_net_flux_1d_cond_slope"] = net_fx[idx1]

    # ---------- 2D edge features (conductance downhill) ----------
    in_wl, out_wl, in_fx, out_fx, net_fx, in_ws, out_ws = edge_in_out_aggs(wl2, src2, dst2, w2_cond, n2)
    d2["e_in_wmean_wl_2d_cond"]  = in_wl[idx2]
    d2["e_out_wmean_wl_2d_cond"] = out_wl[idx2]
    d2["e_in_flux_2d_cond"]      = in_fx[idx2]
    d2["e_out_flux_2d_cond"]     = out_fx[idx2]
    d2["e_net_flux_2d_cond"]     = net_fx[idx2]
    d2["e_in_wsum_2d_cond"]      = in_ws[idx2]
    d2["e_out_wsum_2d_cond"]     = out_ws[idx2]

    # optional: include variant that mixes in static edge slope magnitude
    in_wl, out_wl, in_fx, out_fx, net_fx, in_ws, out_ws = edge_in_out_aggs(wl2, src2, dst2, w2_cond_slope_static, n2)
    d2["e_in_flux_2d_cond_slope"]  = in_fx[idx2]
    d2["e_out_flux_2d_cond_slope"] = out_fx[idx2]
    d2["e_net_flux_2d_cond_slope"] = net_fx[idx2]

    # ---------- coupling features ----------
    wl_1d_map = dict(zip(d1["node_idx"].to_numpy(), d1["water_level"].to_numpy()))
    wl_2d_map = dict(zip(d2["node_idx"].to_numpy(), d2["water_level"].to_numpy()))

    def mean_coupled_wl(node, coupling_map, other_wl_map):
        partners = coupling_map.get(node, [])
        if not partners:
            return 0.0
        vals = [other_wl_map.get(p, np.nan) for p in partners]
        vals = [v for v in vals if not np.isnan(v)]
        if not vals:
            return 0.0
        return float(np.mean(vals))

    d1["wl_coupled_mean_2d"] = [mean_coupled_wl(n, cpl_1d_to_2d, wl_2d_map) for n in d1["node_idx"].to_numpy()]
    d2["wl_coupled_mean_1d"] = [mean_coupled_wl(n, cpl_2d_to_1d, wl_1d_map) for n in d2["node_idx"].to_numpy()]

    d1["node_type"] = "1d"
    d2["node_type"] = "2d"
    return d1, d2

In [512]:
d1_t, d2_t = snapshot_at_t(9)
print(d2_t.isnull().sum())

timestep                    0
node_idx                    0
rainfall                    0
water_level                 0
water_volume                0
position_x                  0
position_y                  0
area                        0
roughness                   0
min_elevation               0
elevation                   0
aspect                      0
curvature                   0
flow_accumulation           0
e_in_wmean_wl_2d_cond       0
e_out_wmean_wl_2d_cond      0
e_in_flux_2d_cond           0
e_out_flux_2d_cond          0
e_net_flux_2d_cond          0
e_in_wsum_2d_cond           0
e_out_wsum_2d_cond          0
e_in_flux_2d_cond_slope     0
e_out_flux_2d_cond_slope    0
e_net_flux_2d_cond_slope    0
wl_coupled_mean_1d          0
node_type                   0
dtype: int64


In [513]:
d1_t.describe()

,timestep,node_idx,water_level,inlet_flow,position_x,position_y,depth,invert_elevation,surface_elevation,base_area,e_in_wmean_wl_1d_cond,e_out_wmean_wl_1d_cond,e_in_flux_1d_cond,e_out_flux_1d_cond,e_net_flux_1d_cond,e_in_wsum_1d_cond,e_out_wsum_1d_cond,e_in_flux_1d_cond_slope,e_out_flux_1d_cond_slope,e_net_flux_1d_cond_slope,wl_coupled_mean_2d
count,17.0,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,1.700000e+01,17.000000,17.000000,17.000000,17.000000,1.700000e+01,17.000000
mean,9.0,8.000000,309.251718,1.289824,802415.905294,349504.593529,4.402586,308.029765,312.432353,14.037647,215.078293,285.057983,0.929199,0.929199,3.856771e-08,0.078303,0.078303,0.078636,0.078636,-3.615723e-09,295.499702
std,0.0,5.049752,16.783005,2.123117,138.201853,307.149574,2.642638,17.540825,17.754943,9.812038,143.465988,74.830780,3.169165,2.229522,4.580458e+00,0.112464,0.065094,0.214580,0.181213,3.165570e-01,77.800261
min,9.0,0.000000,288.143100,0.000000,802092.300000,349133.200000,0.470001,286.600000,287.070000,0.000000,0.000000,0.000000,-1.782119,-1.782119,-7.086263e+00,0.000000,0.000000,-0.075419,-0.075419,-5.773646e-01,0.000000
25%,9.0,4.000000,298.540860,0.000000,802420.060000,349236.900000,2.617004,296.613000,300.680000,12.560000,0.000000,294.603180,-0.305021,-0.305021,-1.811340e+00,0.000000,0.024462,-0.018878,-0.018878,-1.100849e-01,301.441467
50%,9.0,8.000000,305.439850,0.000000,802465.600000,349488.600000,4.498993,304.333000,307.150000,12.560000,298.524109,299.175934,0.000000,0.061841,-6.374074e-01,0.029797,0.066779,0.000000,0.001450,-9.983317e-03,307.661224
75%,9.0,12.000000,313.185460,1.827552,802502.000000,349713.560000,5.842010,312.250000,315.190000,12.560000,304.969269,310.627930,0.061841,1.679027,1.284068e+00,0.093624,0.099429,0.001450,0.083745,6.772175e-02,315.662201
max,9.0,16.000000,347.952450,6.243048,802565.400000,350028.900000,9.902008,348.426000,350.850000,50.240000,333.224335,347.940765,10.645647,5.341503,1.136912e+01,0.421772,0.210210,0.661110,0.577365,6.867659e-01,349.204254


In [514]:
d2_t.describe()

,timestep,node_idx,rainfall,water_level,water_volume,position_x,position_y,area,roughness,min_elevation,elevation,aspect,curvature,flow_accumulation,e_in_wmean_wl_2d_cond,e_out_wmean_wl_2d_cond,e_in_flux_2d_cond,e_out_flux_2d_cond,e_net_flux_2d_cond,e_in_wsum_2d_cond,e_out_wsum_2d_cond,e_in_flux_2d_cond_slope,e_out_flux_2d_cond_slope,e_net_flux_2d_cond_slope,wl_coupled_mean_1d
count,3716.0,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3.716000e+03,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3716.000000,3.716000e+03,3716.000000
mean,9.0,1857.500000,0.109645,322.292484,55.083686,802776.825003,349535.410425,609.391920,0.058643,321.855398,323.160783,184.890633,1.010315e-04,1.513724,315.878235,314.920258,4.198404,4.198404,0.000000,2.047847,2.047847,0.665318,0.665318,1.231872e-08,1.332572
std,0.0,1072.861128,0.006242,14.381728,252.269045,533.101760,400.572766,116.126195,0.018620,14.499653,14.497523,115.095365,3.164418e-04,1.407565,50.926655,45.858620,7.998600,8.510644,9.937724,2.129711,2.034726,1.994864,2.149118,2.651317e+00,20.296836
min,9.0,0.000000,0.000000,293.590820,0.083679,801651.560000,348532.440000,0.000000,0.013000,293.149200,293.812500,-1.000000,3.651328e-31,1.000000,0.000000,0.000000,-3.100985,-1.334589,-105.272476,0.000000,0.000000,-0.555233,-0.040898,-3.633611e+01,0.000000
25%,9.0,928.750000,0.110000,311.266838,3.105634,802326.560000,349257.440000,625.000000,0.060000,310.723788,313.054688,78.087479,2.897490e-06,1.000000,312.203270,308.657272,0.167932,0.179251,-1.337094,0.541708,0.571715,0.003110,0.003457,-9.053321e-02,0.000000
50%,9.0,1857.500000,0.110000,321.978683,6.502419,802776.560000,349532.440000,625.000000,0.060000,321.708250,322.812500,195.151230,1.687669e-05,1.000000,322.713379,320.481171,1.071216,1.037484,-0.004196,1.318773,1.375010,0.045075,0.043685,-1.309376e-04,0.000000
75%,9.0,2786.250000,0.110000,329.830452,16.928801,803224.340000,349832.440000,625.000000,0.060000,329.589115,330.781250,291.648420,6.902532e-05,1.000000,330.649834,328.387489,4.682490,4.413425,1.302160,2.880567,2.882543,0.391449,0.359738,9.053762e-02,0.000000
max,9.0,3715.000000,0.110000,360.006805,2992.147949,803926.560000,350382.440000,1218.562400,0.100000,359.844600,360.218750,359.929350,5.249937e-03,31.000000,360.005280,359.256317,98.096016,130.218369,98.094589,14.100366,16.960590,29.385792,39.753780,2.878627e+01,347.952450


In [515]:
def build_one_step_tables_with_lags(t0, t1):
    rows1 = []
    rows2 = []

    prev1 = None
    prev2 = None

    for t in range(t0, t1 + 1):
        d1_t, d2_t = snapshot_at_t(t)
        d1_tp1, d2_tp1 = snapshot_at_t(t + 1)

        y1 = d1_tp1[["node_idx", "water_level"]].rename(columns={"water_level": "y_next"})
        y2 = d2_tp1[["node_idx", "water_level"]].rename(columns={"water_level": "y_next"})

        m1 = d1_t.merge(y1, on="node_idx", how="inner")
        m2 = d2_t.merge(y2, on="node_idx", how="inner")

        if prev1 is not None:
            p1 = prev1[["node_idx", "water_level"]].rename(columns={"water_level": "wl_lag1"})
            m1 = m1.merge(p1, on="node_idx", how="left")
            m1["dwl_1"] = m1["water_level"] - m1["wl_lag1"]
        else:
            m1["wl_lag1"] = np.nan
            m1["dwl_1"] = np.nan

        if prev2 is not None:
            p2 = prev2[["node_idx", "water_level"]].rename(columns={"water_level": "wl_lag1"})
            m2 = m2.merge(p2, on="node_idx", how="left")
            m2["dwl_1"] = m2["water_level"] - m2["wl_lag1"]
        else:
            m2["wl_lag1"] = np.nan
            m2["dwl_1"] = np.nan

        m1["cpl_wl_diff_mean"] = m1["wl_coupled_mean_2d"] - m1["water_level"]
        m1["cpl_wl_diff_abs"]  = np.abs(m1["cpl_wl_diff_mean"])

        m2["cpl_wl_diff_mean"] = m2["water_level"] - m2["wl_coupled_mean_1d"]
        m2["cpl_wl_diff_abs"]  = np.abs(m2["cpl_wl_diff_mean"])

        m1["timestep"] = t
        m2["timestep"] = t

        rows1.append(m1)
        rows2.append(m2)

        prev1 = d1_t
        prev2 = d2_t

    return pd.concat(rows1, ignore_index=True), pd.concat(rows2, ignore_index=True)


In [516]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

T_TRAIN_END = 80
T_VAL_START = 81
T_VAL_END = 92

train1, train2 = build_one_step_tables_with_lags(0, T_TRAIN_END)
val1, val2     = build_one_step_tables_with_lags(T_VAL_START, T_VAL_END)

# 1D target = delta
train1["y_delta"] = train1["y_next"] - train1["water_level"]
val1["y_delta"]   = val1["y_next"] - val1["water_level"]

FEATS_1D = [
    "water_level", "wl_lag1", "dwl_1", "inlet_flow",
    "invert_elevation", "surface_elevation", "depth", "base_area",
    "e_in_flux_1d_cond", "e_out_flux_1d_cond", "e_net_flux_1d_cond",
    "e_in_wsum_1d_cond", "e_out_wsum_1d_cond",
    "wl_coupled_mean_2d", "cpl_wl_diff_mean", "cpl_wl_diff_abs",
]

FEATS_2D = [
    "water_level", "wl_lag1", "dwl_1", "rainfall", "water_volume",
    "elevation", "roughness", "flow_accumulation", "curvature",
    "e_in_flux_2d_cond", "e_out_flux_2d_cond", "e_net_flux_2d_cond",
    "e_in_wsum_2d_cond", "e_out_wsum_2d_cond",
    "wl_coupled_mean_1d", "cpl_wl_diff_mean", "cpl_wl_diff_abs",
]

FEATS_2D = [c for c in FEATS_2D if c != "water_volume"]
FEATS_1D = [c for c in FEATS_1D if c != "inlet_flow"]
X1_tr = train1[FEATS_1D].fillna(0.0).to_numpy(np.float32)
y1_tr = train1["y_delta"].to_numpy(np.float32)
X1_va = val1[FEATS_1D].fillna(0.0).to_numpy(np.float32)
y1_va = val1["y_delta"].to_numpy(np.float32)

X2_tr = train2[FEATS_2D].fillna(0.0).to_numpy(np.float32)
y2_tr = train2["y_next"].to_numpy(np.float32)
X2_va = val2[FEATS_2D].fillna(0.0).to_numpy(np.float32)
y2_va = val2["y_next"].to_numpy(np.float32)

# 1D: non-linear model
m1 = HistGradientBoostingRegressor(
    max_depth=3,
    learning_rate=0.05,
    max_iter=500,
    l2_regularization=1.0,
    random_state=6,
)

# 2D: keep simple ridge for now
m2 = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("ridge", Ridge(alpha=10.0, random_state=6)),
])

m1.fit(X1_tr, y1_tr)
m2.fit(X2_tr, y2_tr)

p1_delta = m1.predict(X1_va).astype(np.float32)
p1_next  = val1["water_level"].to_numpy(np.float32) + p1_delta
rmse1 = root_mean_squared_error(val1["y_next"].to_numpy(np.float32), p1_next)

p2 = m2.predict(X2_va).astype(np.float32)
rmse2 = root_mean_squared_error(y2_va, p2)

print(f"[2D] one-step RMSE: {rmse2:.6f} | n={len(y2_va)}")
print(f"[1D] one-step RMSE (delta model): {rmse1:.6f} | n={len(p1_next)}")
print("2D feats:", len(FEATS_2D), "1D feats:", len(FEATS_1D))


[2D] one-step RMSE: 0.006956 | n=44592
[1D] one-step RMSE (delta model): 0.029190 | n=204
2D feats: 16 1D feats: 15
